In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("ANAC CSV to Parquet") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.shuffle.partitions", "200") \
    .getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/16 15:06:47 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/04/16 15:06:48 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/04/16 15:06:48 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
26/04/16 15:06:48 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.
26/04/16 15:06:48 WARN Utils: Service 'SparkUI' could not bind on port 4043. Attempting port 4044.


In [2]:
df = spark.read.parquet("hdfs:///user/team12/project/warehouse/dm/anac_parquet")

26/04/16 15:06:51 WARN DomainSocketFactory: The short-circuit local reads feature cannot be used because libhadoop cannot be loaded.


In [3]:
df.printSchema()
print("Rows:", df.count())
print("Columns:", len(df.columns))

root
 |-- id_basica: double (nullable = true)
 |-- id_empresa: integer (nullable = true)
 |-- sg_empresa_icao: string (nullable = true)
 |-- sg_empresa_iata: string (nullable = true)
 |-- nm_empresa: string (nullable = true)
 |-- nm_pais: string (nullable = true)
 |-- ds_tipo_empresa: string (nullable = true)
 |-- nr_voo: integer (nullable = true)
 |-- nr_singular: string (nullable = true)
 |-- id_di: integer (nullable = true)
 |-- cd_di: string (nullable = true)
 |-- ds_di: string (nullable = true)
 |-- ds_grupo_di: string (nullable = true)
 |-- dt_referencia: date (nullable = true)
 |-- nr_ano_referencia: integer (nullable = true)
 |-- nr_semestre_referencia: integer (nullable = true)
 |-- nm_semestre_referencia: string (nullable = true)
 |-- nr_trimestre_referencia: integer (nullable = true)
 |-- nm_trimestre_referencia: string (nullable = true)
 |-- nr_mes_referencia: integer (nullable = true)
 |-- nm_mes_referencia: string (nullable = true)
 |-- nr_semana_referencia: integer (null

Rows: 22120241
Columns: 111


In [4]:
df.show(5, truncate=False)

26/04/16 15:08:36 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-----------+----------+---------------+---------------+----------------------+-------+------------------------+------+-----------+-----+-----+--------------------------+-----------+-------------+-----------------+----------------------+----------------------+-----------------------+-----------------------+-----------------+-----------------+--------------------+------------------------+-----------------+---------------------+-------------+-------------+---------------+----------------------+---------------------+-----------------+-------------------+---------------+-------------------+------------------------+------------------------+-------------------------+-------------------------+-------------------+-------------------+----------------------+--------------------------+-------------------+-----------------------+-------------------+--------------+--------------+-------------------------------------------------------------------------+-------------------+------------+--------------

In [5]:
id_cols = ["id_basica", "id_empresa"]
time_cols = [c for c in df.columns if "dt_" in c or "hr_" in c or "nr_ano" in c]
geo_cols = [c for c in df.columns if "origem" in c or "destino" in c or "uf" in c]
num_cols = [
    "nr_passag_pagos",
    "nr_assentos_ofertados",
    "km_distancia",
    "nr_horas_voadas",
    "nr_decolagem"
]

In [15]:
time_cols

['dt_referencia',
 'nr_ano_referencia',
 'nr_ano_mes_referencia',
 'hr_partida_real',
 'dt_partida_real',
 'nr_ano_partida_real',
 'nr_ano_mes_partida_real',
 'hr_chegada_real',
 'dt_chegada_real',
 'nr_ano_chegada_real',
 'nr_ano_mes_chegada_real',
 'dt_sistema']

In [16]:
geo_cols

['id_aerodromo_origem',
 'sg_icao_origem',
 'sg_iata_origem',
 'nm_aerodromo_origem',
 'nm_municipio_origem',
 'sg_uf_origem',
 'nm_regiao_origem',
 'nm_pais_origem',
 'nm_continente_origem',
 'id_aerodromo_destino',
 'sg_icao_destino',
 'sg_iata_destino',
 'nm_aerodromo_destino',
 'nm_municipio_destino',
 'sg_uf_destino',
 'nm_regiao_destino',
 'nm_pais_destino',
 'nm_continente_destino',
 'nr_escala_destino']

In [17]:
num_cols

['nr_passag_pagos',
 'nr_assentos_ofertados',
 'km_distancia',
 'nr_horas_voadas',
 'nr_decolagem']

In [14]:
from pyspark.sql import functions as F

missing_table = df.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in df.columns
]).toPandas().T.reset_index()

missing_table.columns = ["column_name", "missing_values"]

missing_table

,column_name,missing_values
0,id_basica,0
1,id_empresa,0
2,sg_empresa_icao,0
3,sg_empresa_iata,559947
4,nm_empresa,0
...,...,...
106,id_arquivo,0
107,nm_arquivo,417106
108,nr_linha,356383
109,dt_sistema,0


In [9]:
from pyspark.sql import functions as F

threshold = df.count() * 0.3

valid_cols = [
    c for c in df.columns
    if df.filter(F.col(c).isNull()).count() < threshold
]

In [11]:
len(valid_cols)

109

In [13]:
len(df.columns)

111